In [23]:
!pip install voyageai pinecone-client



In [24]:
import os

os.environ["VOYAGE_API_KEY"] = "pa-pApRGVtFNrbvOBv4r8SGiLg7tARslWUj_sWtwD_dprG"
os.environ["PINECONE_API_KEY"] = "pcsk_3MYtVT_2W6wB82nAeGKxh6GszsLYXv8D6wTH76qj5HgHb5tmi1uNsq5EKKta6aFF8kX8x3"

In [25]:
pip install --upgrade pymupdf


Note: you may need to restart the kernel to use updated packages.


In [26]:
import fitz  # PyMuPDF
# Open the PDF file
doc = fitz.open('/Users/billclinton/Downloads/EMIS-User-Manual-Pre-Primary.pdf')

# Initialize a string to hold the extracted text
pdf_text = ""

# Iterate through each page and extract text
for page in doc:
    pdf_text += page.get_text()

# Close the document
doc.close()

# Display the extracted text
print(pdf_text)


APRIL
2023
(Pre-Primary Schools Version)
USER MANUAL FOR 
PRE-PRIMARY INSTITUTIONS
Education Management Information System (EMIS)
EMIS CONSULTANT:
SMS ONE (U) LTD – WWW.SMSONE.CO.UG
SUPPORTED BY:
1
Acronyms / Abbreviations
Acronym	
            Term
AO                          
Accounting Officer
API                          
Application Programming Interface
BTVET 
             Business, Technical and Vocational Education and Training
CEMIS 
             Central Education Management Information System
DEMIS 
             District Education Management Information System
DEO                          
District Education Officer
DES                          
Directorate of Education Standards
DHI                     
 District Health Inspector
DIS                          
District Inspectorate of Schools
EIA                          
Education Institution Administrator
EISU                          
Education Institution Sub User
EMIS               
 Education Management Information Syste

In [27]:
def chunk_text(text, max_length=1000):
    """
    Splits the text into chunks of specified maximum length.
    """
    return [text[i:i+max_length] for i in range(0, len(text), max_length)]

# Chunk the extracted text
all_chunks = chunk_text(pdf_text)


In [28]:
import sys
print(sys.executable)


/opt/anaconda3/envs/ml/bin/python


In [29]:
import sys
!{sys.executable} -m pip install voyageai


In [30]:
import voyageai

# Initialize the Voyage AI client
vc = voyageai.Client()  # Assumes VOYAGE_API_KEY is set in your environment

# Generate embeddings for the text chunks
response = vc.embed(
    texts=chunks,
    model="voyage-3-large",  # Replace with your chosen model
    input_type="document",
    truncation=True
)

# Extract the embeddings from the response
embeddings = response.embeddings

RateLimitError: You have not yet added your payment method in the billing page and will have reduced rate limits of 3 RPM and 10K TPM. To unlock our standard rate limits, please add a payment method in the billing page for the appropriate organization in the user dashboard (https://dashboard.voyageai.com/). Even with payment methods entered, the free tokens (200M tokens for Voyage series 3) will still apply. After adding a payment method, you should see your rate limits increase after several minutes. See our pricing docs (https://docs.voyageai.com/docs/pricing) for the free tokens for your model.

In [32]:
import voyageai
import time
import os

# Initialize the Voyage AI client
vc = voyageai.Client()  # Assumes VOYAGE_API_KEY is set in your environment

# Function to process chunks with rate limiting
def process_with_rate_limit(chunks, batch_size=2):
    all_embeddings = []
    total_batches = (len(chunks) + batch_size - 1) // batch_size
    
    print(f"Starting to process {len(chunks)} chunks in {total_batches} batches...")
    
    # Process chunks in smaller batches
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i+batch_size]
        current_batch = i//batch_size + 1
        
        print(f"Processing batch {current_batch}/{total_batches}...")
        
        # Generate embeddings for the current batch
        response = vc.embed(
            texts=batch,
            model="voyage-3-large",  # Using your chosen model
            input_type="document",
            truncation=True
        )
        
        # Extract embeddings from response
        all_embeddings.extend(response.embeddings)
        
        # Wait to respect rate limits (3 RPM for free tier)
        if i + batch_size < len(chunks):
            print(f"Processed batch {current_batch}/{total_batches}. Waiting to respect rate limits...")
            time.sleep(20)  # Wait 20 seconds between batches
    
    # Notification when complete
    print("\n✅ EMBEDDING PROCESS COMPLETE!")
    
    # Try to send a system notification if on macOS
    try:
        os.system('osascript -e \'display notification "Embedding process complete!" with title "Voyage AI Embeddings"\' &> /dev/null')
    except:
        pass
    
    # Try to send a system notification if on Linux
    try:
        os.system('notify-send "Voyage AI Embeddings" "Embedding process complete!"')
    except:
        pass
    
    return all_embeddings

# Generate embeddings with rate limiting
embeddings = process_with_rate_limit(chunks)
print(f"Successfully generated {len(embeddings)} embeddings.")

Starting to process 56 chunks in 28 batches...
Processing batch 1/28...
Processed batch 1/28. Waiting to respect rate limits...
Processing batch 2/28...
Processed batch 2/28. Waiting to respect rate limits...
Processing batch 3/28...
Processed batch 3/28. Waiting to respect rate limits...
Processing batch 4/28...
Processed batch 4/28. Waiting to respect rate limits...
Processing batch 5/28...
Processed batch 5/28. Waiting to respect rate limits...
Processing batch 6/28...
Processed batch 6/28. Waiting to respect rate limits...
Processing batch 7/28...
Processed batch 7/28. Waiting to respect rate limits...
Processing batch 8/28...
Processed batch 8/28. Waiting to respect rate limits...
Processing batch 9/28...
Processed batch 9/28. Waiting to respect rate limits...
Processing batch 10/28...
Processed batch 10/28. Waiting to respect rate limits...
Processing batch 11/28...
Processed batch 11/28. Waiting to respect rate limits...
Processing batch 12/28...
Processed batch 12/28. Waiting t

sh: notify-send: command not found


In [33]:
import sys
!{sys.executable} -m pip uninstall pinecone-client -y


In [34]:
!{sys.executable} -m pip install pinecone


In [35]:
import pinecone
import os

# Create an instance of the Pinecone class instead of using init()
pc = pinecone.Pinecone(
    api_key=os.environ["PINECONE_API_KEY"]
)

# Define your index name
index_name = "voyageai-emis-index"

# Create the index if it doesn't exist
if index_name not in pc.list_indexes().names():
    # Create the index with a ServerlessSpec (required in newer Pinecone SDK)
    pc.create_index(
        name=index_name, 
        dimension=1024, 
        metric="cosine",
        spec=pinecone.ServerlessSpec(
            cloud='aws',
            region='us-east-1'  # Adjust region as needed
        )
        # Alternatively, you could use PodSpec for pod-based deployment:
        # spec=pinecone.PodSpec(
        #     environment='gcp-starter',  # Adjust environment as needed
        #     replicas=1,
        #     pods=1
        # )
    )

# Connect to the index
index = pc.Index(index_name)

In [ ]:
# Prepare data for upsert
ids = [f"chunk-{i}" for i in range(len(all_chunks))]
metadata = [{"text": chunk} for chunk in all_chunks]
vectors = list(zip(ids, embeddings, metadata))

# Upsert the vectors into Pinecone
index.upsert(vectors=vectors)


In [36]:
# Describe index statistics
index_stats = index.describe_index_stats()
print(index_stats)


{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 56}},
 'total_vector_count': 56,
 'vector_type': 'dense'}


In [37]:
query = "what is emis"
query_embedding = vc.embed(
    texts=[query],
    model="voyage-3-large",
    input_type="document",
    truncation=True
).embeddings[0]


In [ ]:
results = index.query(
    vector=query_embedding,
    top_k=5,
    include_metadata=True
)
for match in results['matches']:
    print(f"ID: {match['id']}, Score: {match['score']}, Metadata: {match['metadata']}")
